# Follow-up: bounded compute scaling

This companion runs **only after Notebook 04 finishes successfully and its detached worker exits**. It uses the completed plan's first seed and common baseline configuration to measure ViT and semantic-local attention at 256, 1,024 and 4,096 patch tokens. These are synthetic RGB compute measurements, not quality evaluations or a scaling-law fit.

**Inputs:** `artifacts/notebook_04/semantic-followup-001/{plan.json,budget_clock.json,pilot_status.json,launch.json}` and the unchanged tagged notebook exports frozen by that plan. **Outputs:** `scale_screen.json`, per-case JSON records in `scale_screen/`, native Matplotlib figures and their PNG files, and `scaling_companion_status.json` under that same artifact directory. No original source, completed learning result or checkpoint is modified.

**Runtime:** at most a 180-second measurement budget, with three warmup and five measured calls per case; individual cases also have a 30-second guard. It refuses to launch if the main experiment is incomplete or its worker still exists. With less than 180 seconds left on the **original** three-hour clock, it records an honest skip. An external timeout at that original deadline must wrap execution because nested per-case alarms are not a substitute for a process-level deadline.

**Resume:** run this notebook top-to-bottom using the same run ID, source and plan before the original deadline. The frozen screen checks and reuses completed per-case records. After the deadline, existing records remain viewable and no new measurement starts. An interrupted case may need to rerun from its beginning; each case is under 30 seconds and is not a learning run. No held-out images are used.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import json, copy, time, subprocess
import nbformat
from IPython.display import display, Markdown
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa")
RUN_ID = "semantic-followup-001"
ARTIFACT = ROOT / "artifacts/notebook_04" / RUN_ID
MPL_CACHE = ARTIFACT / "matplotlib_cache"
MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPL_CACHE)
for cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if cell.cell_type == "code" and "ahr-export" in cell.get("metadata", {}).get("tags", []):
        exec(compile(cell.source, "bootstrap#ahr-export", "exec"))
install_notebook_package(ROOT)
import importlib.util

def load_scaling_export(name):
    path = ROOT / "experiments/04_followup" / (name + ".ipynb")
    loader = TaggedNotebookLoader(path)
    spec = importlib.util.spec_from_loader("bounded_scaling_" + name, loader)
    module = importlib.util.module_from_spec(spec)
    loader.exec_module(module)
    return module

WORKFLOW = load_scaling_export("workflow")
ENGINEERING = load_scaling_export("engineering")
from fmm_jepa.training import atomic_json, digest_file, run_writer


In [ ]:
def scaling_prerequisites(root, artifact):
    """CPU-only prerequisite and source validation; never starts a CUDA workload."""
    root, artifact = Path(root), Path(artifact)
    plan = json.loads((artifact / "plan.json").read_text())
    clock = json.loads((artifact / "budget_clock.json").read_text())
    status = json.loads((artifact / "pilot_status.json").read_text())
    launch = json.loads((artifact / "launch.json").read_text())
    WORKFLOW.check_followup_plan(root, plan)
    if status.get("status") != "COMPLETED":
        raise RuntimeError("Main Notebook 04 has not completed; do not run scaling concurrently.")
    if plan["run_id"] != artifact.name:
        raise ValueError("Plan run identity differs from artifact folder")
    if plan["budget"]["clock"] != clock:
        raise ValueError("Original budget clock differs from the frozen plan")
    deadline, started = float(clock["deadline_unix"]), float(clock["started_unix"])
    if not 0 < deadline - started <= 10800.0001:
        raise ValueError("Invalid original three-hour budget")
    if float(status["updated_unix"]) > deadline:
        raise ValueError("Main completion was after the authorized deadline")
    session = str(launch["session"])
    probe = subprocess.run(["tmux", "has-session", "-t", "=" + session], capture_output=True, text=True)
    if probe.returncode == 0:
        raise RuntimeError("Main detached worker still exists: " + session)
    if probe.returncode not in (1,):
        raise RuntimeError("Could not establish worker absence: " + probe.stderr.strip())
    first_seed = min(int(arm["seed"]) for arm in plan["arms"])
    first = [arm for arm in plan["arms"] if int(arm["seed"]) == first_seed]
    if plan["factor"] == "ema":
        vit = next(arm for arm in first if arm["kind"] == "vit" and arm["condition"] == "ema0996")
        semantic = next(arm for arm in first if arm["kind"] == "semantic" and arm["condition"] == "ema0996")
    elif plan["factor"] == "feature_rank":
        vit = next(arm for arm in first if arm["kind"] == "vit" and arm["condition"] == "reference")
        semantic = next(arm for arm in first if arm["kind"] == "semantic" and arm["condition"] == "features32")
    else:
        raise ValueError("Unrecognized registered mathematical factor")
    config = copy.deepcopy(semantic["config"])
    if vit["config"] != semantic["config"]:
        raise ValueError("Baseline ViT and semantic arms do not share a common configuration")
    if config.get("dtype") != "float32" or config["device"] != "cuda:0":
        raise ValueError("Expected registered deterministic FP32 GPU0 configuration")
    if config["evaluation"].get("heldout_enabled", False):
        raise ValueError("Held-out evaluation is forbidden in this companion")
    return {"plan": plan, "clock": clock, "config": config, "deadline_unix": deadline,
            "remaining_seconds": deadline - time.time(), "main_worker_session": session,
            "baseline_arm_ids": [vit["id"], semantic["id"]], "first_seed": first_seed,
            "source_validation": "PASS", "main_completion": "COMPLETED", "worker_absent": True}


def execute_scaling_companion(root, artifact):
    """Respect the original deadline; no quality-selection decisions are made here."""
    root, artifact = Path(root), Path(artifact)
    prerequisites = scaling_prerequisites(root, artifact)
    path = artifact / "scale_screen.json"
    status_path = artifact / "scaling_companion_status.json"
    common = {"original_deadline_unix": prerequisites["deadline_unix"],
              "baseline_arm_ids": prerequisites["baseline_arm_ids"],
              "first_seed": prerequisites["first_seed"],
              "plan_sha256": digest_file(artifact / "plan.json"),
              "frozen_source_validation": "PASS", "main_worker_absent": True,
              "quality_evidence": False, "official_test_status": "NOT_RUN"}
    if path.exists():
        # Presentation-only reuse: never rewrite the completed measurement summary.
        existing = json.loads(path.read_text())
        expected_config = copy.deepcopy(prerequisites["config"])
        expected_config["training"].update(updates=100, checkpoint_every=4)
        expected_config["evaluation"]["heldout_enabled"] = False
        if existing["identity"]["config"] != expected_config:
            raise ValueError("Completed scale cache configuration differs from the plan")
        if existing["identity"].get("engineering_export_sha256") != ENGINEERING._engineering_export_hash(root):
            raise ValueError("Completed scale cache source identity changed")
        if existing["identity"].get("source_hashes") != ENGINEERING.definition_hashes(root):
            raise ValueError("Completed scale cache upstream source identity changed")
        if ENGINEERING._engineering_hash(existing["identity"]) != existing["identity_hash"]:
            raise ValueError("Completed scale cache identity checksum invalid")
        if existing["identity"].get("token_counts") != [256, 1024, 4096]:
            raise ValueError("Completed scale cache token counts differ")
        expected_keys = {f"{tokens}_{scope}_{kind}" for tokens in (256, 1024, 4096)
                         for scope in ("encoder_forward", "full_jepa_update") for kind in ("vit", "semantic")}
        if len(existing["rows"]) != 12 or {row["case_key"] for row in existing["rows"]} != expected_keys:
            raise ValueError("Completed scale cache does not contain all twelve unique cases")
        if existing.get("status") != "PASS":
            raise ValueError("Existing scale summary is incomplete; refusing new measurements during presentation")
        for row in existing["rows"]:
            if row.get("status") != "PASS" or row.get("warmup") != 3 or row.get("repeats") != 5:
                raise ValueError("Completed scale cache has an invalid case")
            case = json.loads((artifact / "scale_screen" / (row["case_key"] + ".json")).read_text())
            if case != row or row["identity_hash"] != existing["identity_hash"]:
                raise ValueError("Completed scale cache case differs from its summary")
        status = json.loads(status_path.read_text())
        if status.get("status") != "PASS" or status.get("result_sha256") != digest_file(path):
            raise ValueError("Completed scale status does not certify this result")
        if status.get("finished_unix", float("inf")) > prerequisites["deadline_unix"]:
            raise ValueError("Saved scaling completion exceeded the original deadline")
        display(Markdown("Using all **12 validated completed cases**. No GPU measurements are started and measurement files are preserved."))
        return existing, status
    if os.environ.get("FMM_SCALING_CACHE_ONLY") == "1":
        raise RuntimeError("Cache-only presentation requested but scale_screen.json is missing")
    if time.time() >= prerequisites["deadline_unix"] - 180:
        status = {**common, "status": "SKIPPED_INSUFFICIENT_ORIGINAL_BUDGET",
                  "remaining_seconds": max(0.0, prerequisites["deadline_unix"] - time.time()),
                  "required_seconds": 180, "updated_unix": time.time(),
                  "reason": "No new measurement started; fewer than 180 seconds remain on the original clock."}
        if path.exists():
            existing = json.loads(path.read_text())
            expected_config = ENGINEERING._engineering_configuration(root, prerequisites["config"])
            expected_config["training"]["updates"] = 100
            if existing["identity"]["config"] != expected_config:
                raise ValueError("Existing scale measurements use a different plan configuration")
            if existing["identity"]["engineering_export_sha256"] != ENGINEERING._engineering_export_hash(root):
                raise ValueError("Existing scale source identity differs")
            if existing["identity"].get("token_counts") != [256, 1024, 4096]:
                raise ValueError("Existing scale token counts differ")
            status["existing_measurements_preserved"] = True
            status["existing_result_sha256"] = digest_file(path)
            atomic_json(status_path, status)
            return existing, status
        atomic_json(status_path, status)
        return None, status
    with run_writer(artifact / "scaling_companion"):
        # Recheck immediately before the only GPU workload.
        prerequisites = scaling_prerequisites(root, artifact)
        if prerequisites["remaining_seconds"] < 180:
            status = {**common, "status": "SKIPPED_INSUFFICIENT_ORIGINAL_BUDGET",
                      "remaining_seconds": max(0.0, prerequisites["remaining_seconds"]),
                      "required_seconds": 180, "updated_unix": time.time()}
            atomic_json(status_path, status)
            return None, status
        atomic_json(status_path, {**common, "status": "RUNNING", "started_unix": time.time()})
        try:
            result = ENGINEERING.run_scale_screen(root, artifact, prerequisites["deadline_unix"],
                config=prerequisites["config"], token_counts=(256, 1024, 4096))
            finished = time.time()
            if finished > prerequisites["deadline_unix"]:
                raise TimeoutError("Scaling returned after original deadline; no in-budget completion claimed")
            WORKFLOW.check_followup_plan(root, prerequisites["plan"])
            status = {**common, "status": result["status"], "finished_unix": finished,
                      "within_original_deadline": True, "wall_seconds": result["wall_seconds"],
                      "result_sha256": digest_file(path)}
            atomic_json(status_path, status)
            return result, status
        except BaseException as exc:
            atomic_json(status_path, {**common, "status": "INTERRUPTED", "updated_unix": time.time(),
                                     "error": type(exc).__name__ + ": " + str(exc),
                                     "per_case_evidence_preserved": True})
            raise


def plot_scaling_result(result, artifact):
    import matplotlib.pyplot as plt
    artifact = Path(artifact)
    if result is None:
        display(Markdown("No scaling measurements were started. The original experiment time limit is preserved."))
        return []
    rows = result["rows"]
    table = ["| Scope | Model | Tokens | Status | Median (ms) | Peak allocation (GiB) | Measured calls |",
             "|---|---|---:|---|---:|---:|---:|"]
    for row in rows:
        median = f"{1000 * row['median_seconds']:.2f}" if row.get("median_seconds") is not None else "—"
        peak = f"{row['peak_allocated_bytes'] / 1024**3:.3f}" if row.get("peak_allocated_bytes") is not None else "—"
        table.append(f"| {row['scope']} | {row['model_id']} | {row['tokens']:,} | {row['status']} | {median} | {peak} | {row.get('repeats', 0)} |")
    display(Markdown("\n".join(table)))
    colors = {"vit": "#0072B2", "semantic": "#D55E00"}
    labels = {"vit": "ViT", "semantic": "Semantic-local"}
    figures = []
    for scope, title in (("encoder_forward", "Encoder forward"), ("full_jepa_update", "Complete JEPA update")):
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
        for kind in ("vit", "semantic"):
            valid = sorted((r for r in rows if r["scope"] == scope and r["model_id"] == kind and r["status"] == "PASS"), key=lambda r:r["tokens"])
            if valid:
                x = [r["tokens"] for r in valid]
                axes[0].plot(x, [r["median_seconds"] * 1000 for r in valid], "o-", color=colors[kind], label=labels[kind])
                axes[1].plot(x, [r["peak_allocated_bytes"] / 1024**3 for r in valid], "o-", color=colors[kind], label=labels[kind])
        for ax in axes:
            ax.set_xscale("log", base=2)
            ax.set_xticks([256, 1024, 4096], labels=["256", "1,024", "4,096"])
            ax.set_xlabel("Patch tokens")
            ax.set_ylim(bottom=0)
            ax.grid(axis="y", alpha=.25)
            if ax.lines:
                ax.legend(frameon=False)
        axes[0].set_ylabel("Median synchronized call (ms)")
        axes[1].set_ylabel("Peak CUDA allocation (GiB)")
        failed = [r for r in rows if r["scope"] == scope and r["status"] != "PASS"]
        subtitle = "Synthetic RGB • batch 2 • FP32 • 3 warmup + 5 timed calls"
        if failed:
            subtitle += "\nOmitted points: " + "; ".join(f"{labels[r['model_id']]} {r['tokens']}: {r['status']}" for r in failed)
        fig.suptitle(title + "\n" + subtitle, fontsize=12)
        target = artifact / ("scaling_" + scope + ".png")
        fig.savefig(target, dpi=180)
        display(fig)
        plt.close(fig)
        figures.append(str(target))
    display(Markdown("Timings exclude preprocessing, transfer, checkpoint I/O and profiler work. Full updates include backward, clipping, optimizer, scheduler and EMA. Peaks include live model/input/optimizer allocations, but exclude CUDA context and reserved allocator memory. Missing or failed cases are omitted from curves, never plotted as zero. These fixed-batch synthetic measurements do not establish useful learning, natural-image accuracy, or asymptotic scaling."))
    atomic_json(artifact / "scaling_figure_manifest.json", {"figures": [{"path": p, "sha256": digest_file(p)} for p in figures],
        "source_result_sha256": digest_file(artifact / "scale_screen.json"), "visual_inspection": "PENDING",
        "quality_evidence": False})
    return figures


## Validate completion and measure within the original clock

This cell starts the bounded scaling work only after all prerequisites pass. Run it through the externally timed controller described above.

In [ ]:
SCALING_RESULT, SCALING_STATUS = execute_scaling_companion(ROOT, ARTIFACT)
display(Markdown("**Scaling status:** " + SCALING_STATUS["status"]))
SCALING_STATUS

## Matched compute measurements

Each architecture uses the same synthetic images, augmentation and masks. Lines connect only measured points; a small grid of token counts does not justify extrapolation.

In [ ]:
SCALING_FIGURES = plot_scaling_result(SCALING_RESULT, ARTIFACT)